In [0]:
import time
import datetime

_data = [
(11114,datetime.datetime.strptime('08:30:00:00',"%H:%M:%S:%f"),"I"),
(11114,datetime.datetime.strptime('10:30:00:00',"%H:%M:%S:%f"),'O'),
(11114,datetime.datetime.strptime('11:30:00:00',"%H:%M:%S:%f"),'I'),
(11114,datetime.datetime.strptime('15:30:00:00',"%H:%M:%S:%f"),'O'),
(11115,datetime.datetime.strptime('09:30:00:00',"%H:%M:%S:%f"),'I'),
(11115,datetime.datetime.strptime('17:30:00:00',"%H:%M:%S:%f"),'O')
]
from pyspark.sql.types import StructType,StructField, TimestampType, LongType, StringType
_schema = StructType([
  StructField('emp_id', LongType(), True),
  StructField('punch_time', TimestampType(), True),
  StructField('flag', StringType(), True)
])
df = spark.createDataFrame(data = _data, schema=_schema)
df.show()


# m2

In [0]:
w = Window.partitionBy("emp_id").orderBy("punch_time")

result = (
    df
    .withColumn("next_time", F.lead("punch_time").over(w))
    .withColumn("next_flag", F.lead("flag").over(w))
    .filter(
        (F.col("flag") == "I") &
        (F.col("next_flag") == "O")
    )
    .withColumn(
        "worked_seconds",
        F.unix_timestamp("next_time")
        -
        F.unix_timestamp("punch_time")
    )
    .groupBy("emp_id")
    .agg(
        (
            F.sum("worked_seconds") / 3600
        ).alias("worked_hours")
    )
)

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

In [0]:
w = Window.partitionBy("emp_id").orderBy("punch_time")

time_df = (
    df.withColumn("next_punch_time",F.lead(F.col("punch_time")).over(w))
    .filter(F.col("flag") == 'I')
    .withColumn("time",
    F.timestamp_diff("HOUR",F.col("punch_time"),F.col("next_punch_time")))
    .groupby("emp_id")
    .agg(
        F.sum("time").alias("total_time")
    )
)
display(time_df)

# Q2. Write a solution to swap the seat id of every two consecutive students. If the number of students is odd, the id of the last student is not swapped.

In [0]:
_data = [(1,'Abbot'),(2,'Doris'),(3,'Emerson'),(4,'Green'),(5,'Jeames')]
_schema = ['id', 'student']

df = spark.createDataFrame(data = _data, schema=_schema)

In [0]:
max_id = df.agg(F.max("id")).collect()[0][0]

df = (
    df.withColumn(
        "id",
        F.when(
            ((F.col("id") % 2) == 1) & (F.col("id") == max_id),
            F.col("id")
        )
        .when(((F.col("id") % 2) == 0), F.col("id") - 1)
        .otherwise(F.col("id") + 1)
    )
    .orderBy("id")
)

df.show()